In [ ]:
%%capture
!pip install -r requirements.txt
from utils import *

wd = WarpDrive()
dataset_type = wd.get_args('dataset_type')

In [ ]:
# Read the data
container = 'solytics-saas-nimbus-uno-uat-s3'

if dataset_type == 'train':
    anom_scores_path = 'poc_anomaly_detection/consumer_banking/outputs/anomaly_aggregated_scores_consumer_banking.parquet'
    tms_path = 'poc_anomaly_detection/consumer_banking/data/tms_data_consumer_banking_train.parquet'
    cluster_desc_path = 'poc_anomaly_detection/consumer_banking/data/cluster_description_mapping_consumer_banking_train.parquet'
    anom_feat_path = 'poc_anomaly_detection/consumer_banking/data/anomaly_feaures_data_consumer_banking_train.parquet'
else:
    anom_scores_path = 'poc_anomaly_detection/consumer_banking/outputs/anomaly_aggregated_scores_consumer_banking_test.parquet'
    tms_path = 'poc_anomaly_detection/consumer_banking/data/tms_data_consumer_banking_test.parquet'
    cluster_desc_path = 'poc_anomaly_detection/consumer_banking/data/cluster_description_mapping_consumer_banking_test.parquet'
    anom_feat_path = 'poc_anomaly_detection/consumer_banking/data/anomaly_feaures_data_consumer_banking_test.parquet'

def read_data(file_path):
    with wd.store.get_object_readable_stream(container, file_path) as stream:
        df = pd.read_parquet(stream)
    return df

df_anom_scores = read_data(anom_scores_path)
df_tms = read_data(tms_path)
df_cluster_desc = read_data(cluster_desc_path)
df_anom_features = read_data(anom_feat_path)

In [ ]:
df_anom_features = df_anom_features.fillna(0)

# Drop duplicated cols
df_tms = df_tms.rename(columns={'ACCOUNT_BALANCE_x' : 'ACCOUNT_BALANCE', 
                                'MONTHLY_INCOME_x' : 'MONTHLY_INCOME', 
                                'NOMINEE_DETAILS_x' : 'NOMINEE_DETAILS', 
                                'ACCOUNT_TENURE_x' : 'ACCOUNT_TENURE', 
                                'BRANCH_x' : 'BRANCH'})

drop_cols = [col for col in df_tms.columns if col.endswith('_y')]
df_tms = df_tms.drop(drop_cols, axis=1, errors='ignore')

In [ ]:
df_scores_temp = df_anom_scores[['PARTY_NUMBER', 'Cluster Label', 'Aggregate Anomaly Score', 'Alert Level', 'Is Anomaly']].copy()
df_anom_features = df_anom_features.rename(columns={'cluster_label' : 'Cluster Label'})

In [ ]:
# Bring the anomaly scores and anomaly flag to df_anom_features and df_tms
df_anom_features = df_anom_features.merge(df_scores_temp, how='left', on=['PARTY_NUMBER', 'Cluster Label'])
df_tms = df_tms.merge(df_scores_temp, how='left', on=['PARTY_NUMBER'])

df_anom_features.shape, df_tms.shape

In [ ]:
cluster_desc_dict = dict(zip(df_anom_features['Cluster Label'], df_anom_features['cluster_desc']))
party_anomaly_dict = dict(zip(df_anom_features['PARTY_NUMBER'], df_anom_features['Is Anomaly']))

In [ ]:
Mahalanobis Distance

For each party:

Mahalanobis measures how far their overall behaviour is from the cluster norm

across all features simultaneously

Interpretation

Low value → typical peer behaviour

High value → unusual combination of behaviours

In [ ]:
from scipy.stats import percentileofscore

feature_cols = [
    col for col in df_anom_features.columns
    if col not in [
        'PARTY_NUMBER', 'Cluster Label', 'cluster_desc',
        'Aggregate Anomaly Score', 'Alert Level', 'Is Anomaly'
    ]
]

# -------------------------------
# Initialise columns
# -------------------------------
df_anom_features['mahalanobis'] = np.nan
df_anom_features['mahalanobis_percentile'] = np.nan
df_anom_features['top_features'] = None
df_anom_features['top_features_contrib_pct'] = None
df_anom_features['top_features_percentile'] = None
df_anom_features['mahalanobis_interpretation'] = None

for col in ['top_features', 'top_features_contrib_pct', 'top_features_percentile']:
    if col in df_anom_features.columns:
        df_anom_features = df_anom_features.drop(columns=col, axis=1, errors='ignore')
    df_anom_features[col] = None

# -------------------------------
# Numeric enforcement + NA handling
# -------------------------------
df_anom_features[feature_cols] = (
    df_anom_features[feature_cols]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
)

# -------------------------------
# Cluster-wise Mahalanobis + explanation
# -------------------------------
for cluster_id, cluster_df in df_anom_features.groupby('Cluster Label'):

    X = cluster_df[feature_cols].values
    mean_vec = X.mean(axis=0)

    cov = np.cov(X, rowvar=False)
    cov += np.eye(cov.shape[0]) * 1e-6
    cov_inv = np.linalg.pinv(cov)

    # Mahalanobis distance
    distances = np.sqrt(np.einsum('ij,jk,ik->i', X - mean_vec, cov_inv, X - mean_vec))

    # Write distances
    df_anom_features.loc[cluster_df.index, 'mahalanobis'] = distances

    # Mahalanobis percentile
    md_percentiles = [
        percentileofscore(distances, d, kind='mean') if len(distances) > 1 else np.nan
        for d in distances
    ]
    df_anom_features.loc[cluster_df.index, 'mahalanobis_percentile'] = np.round(md_percentiles, 2)

    # Feature contributions
    diag_inv = np.diag(cov_inv)
    contributions = (X - mean_vec) ** 2 * diag_inv
    contrib_sum = contributions.sum(axis=1)
    contrib_pct = contributions / np.where(contrib_sum[:, None] == 0, 1, contrib_sum[:, None])

    # Top 3 features
    top_idx = np.argsort(contrib_pct, axis=1)[:, -3:][:, ::-1]
    top_feats = [[feature_cols[i] for i in row_idx] for row_idx in top_idx]
    top_contrib = [list(contrib_pct[i, row_idx] * 100) for i, row_idx in enumerate(top_idx)]
    top_contrib = [[round(v, 2) for v in (contrib_pct[i, row_idx] * 100)] for i, row_idx in enumerate(top_idx)]
    
    # Assign per row
    for i, idx in enumerate(cluster_df.index):
        df_anom_features.at[idx, 'top_features'] = top_feats[i]
        df_anom_features.at[idx, 'top_features_contrib_pct'] = top_contrib[i]

    # Percentiles + interpretation
    for i, idx in enumerate(cluster_df.index):

        feats = top_feats[i]
        contribs = top_contrib[i]
        percentiles = []
        interp_parts = []

        for f, c in zip(feats, contribs):

            feat_idx = feature_cols.index(f)
            val = cluster_df.loc[idx, f]
            peers = cluster_df[f].dropna()

            perc = np.round(percentileofscore(peers, val, kind='mean'), 2) if len(peers) > 1 else np.nan
            percentiles.append(perc)

            deviation = val - mean_vec[feat_idx]
            direction = "higher" if deviation > 0 else "lower"
            desc = feature_desc_map.get(f, f)

            if perc >= 95:
                peer_text = "higher than almost all similar customers"
            elif perc >= 80:
                peer_text = "higher than most similar customers"
            elif perc >= 60:
                peer_text = "above the typical range of similar customers"
            elif perc >= 40:
                peer_text = "within the typical range of similar customers"
            else:
                peer_text = "closer to peers in the typical range"

            interp_parts.append(f"{desc} ({direction}, ~{c:.0f}% contribution, {peer_text})")

        df_anom_features.at[idx, 'top_features_percentile'] = percentiles

        # Mahalanobis interpretation based on raw score (cluster-scaled)
        raw_score = df_anom_features.at[idx, 'mahalanobis']
        cluster_scores = df_anom_features.loc[cluster_df.index, 'mahalanobis'].values
        min_score = cluster_scores.min()
        max_score = cluster_scores.max()
        scaled_score = (raw_score - min_score) / (max_score - min_score + 1e-6)

        if scaled_score <= 0.2:
            md_sentence = f"Mahalanobis distance of {raw_score:.2f} indicates typical behaviour relative to cluster peers."
        elif scaled_score <= 0.4:
            md_sentence = f"Mahalanobis distance of {raw_score:.2f} shows mild deviation from cluster norms."
        elif scaled_score <= 0.6:
            md_sentence = f"Mahalanobis distance of {raw_score:.2f} indicates noticeable deviation from cluster patterns."
        elif scaled_score <= 0.8:
            md_sentence = f"Mahalanobis distance of {raw_score:.2f} indicates strong deviation from cluster norms."
        else:
            md_sentence = f"Mahalanobis distance of {raw_score:.2f} indicates extreme deviation from cluster norms and may warrant further review."

        df_anom_features.at[idx, 'mahalanobis_interpretation'] = (
            md_sentence + " Deviation relative to peers is driven mainly by: " + "; ".join(interp_parts)
        )

# -------------------------------
# Final dataframe
# -------------------------------
select_cols = [
    'PARTY_NUMBER', 'Cluster Label', 'mahalanobis', 'mahalanobis_percentile',
    'top_features', 'top_features_contrib_pct', 'top_features_percentile',
    'mahalanobis_interpretation', 'Is Anomaly'
]

df_db_mahalanobis = df_anom_features[df_anom_features['Is Anomaly'] == True][select_cols]
df_db_mahalanobis['mahalanobis'] = round(df_db_mahalanobis['mahalanobis'], 2)
df_db_mahalanobis = df_db_mahalanobis[['PARTY_NUMBER', 'Cluster Label', 'Is Anomaly',  'mahalanobis', 
                                       'mahalanobis_percentile', 'top_features', 'top_features_contrib_pct',
                                       'top_features_percentile', 'mahalanobis_interpretation']]
df_db_mahalanobis = df_db_mahalanobis.reset_index(drop=True)
df_db_mahalanobis = df_db_mahalanobis.rename(columns={'mahalanobis' : 'Mahalanobis Distance', 'mahalanobis_interpretation' : 'Interpretation', 'mahalanobis_percentile' : 'Mahalanobis Percentile',
                                                      'top_features' : 'Top Features', 'top_features_contrib_pct' : 'Top Features Contribution %', 'top_features_percentile' : 'Top Features Percentile'})

In [ ]:
metrics = []

# Aggregate basic metrics per party
party_agg = df_tms.groupby(['PARTY_NUMBER', 'Cluster Label']).agg(
    total_txn_count=('TXN_REF_NUMBER','count'),
    total_txn_amount=('CURRENCY_AMOUNT','sum'),
    avg_txn_amount=('CURRENCY_AMOUNT','mean'),
    max_txn_amount=('CURRENCY_AMOUNT','max'),
    min_txn_amount=('CURRENCY_AMOUNT','min')
).reset_index()

metric_desc_map = {
    'total_txn_count': 'Transaction volume',
    'total_txn_amount': 'Transaction value',
    'avg_txn_amount': 'Average value',
    'max_txn_amount': 'Largest transaction'
}

# Compute peer-based percentile and z-score
final_metrics = []

for cluster_id, cluster_df in party_agg.groupby('Cluster Label'):
    for metric in ['total_txn_count','total_txn_amount','avg_txn_amount','max_txn_amount']:
        metric_desc = metric_desc_map.get(metric, 'Baseline')
        values = cluster_df[metric].values
        mean_val = np.mean(values)
        std_val = np.std(values)
        for idx, row in cluster_df.iterrows():
            perc = percentileofscore(values, row[metric], kind='rank')
            z = (row[metric] - mean_val)/std_val if std_val>0 else 0
            
            # Dynamic interpretation
            if perc >= 95:
                desc = "very high"
            elif perc >= 75:
                desc = "high"
            elif perc >= 50:
                desc = "above average"
            elif perc >= 25:
                desc = "below average"
            else:
                desc = "very low"
            
            interpretation = (f"This party's {metric_desc.lower()} is {desc} compared to peers in the cluster "
                              f"(percentile={int(perc)}, z-score={z:.2f}, value={row[metric]:.2f})")
            
            final_metrics.append({
                'PARTY_NUMBER': row['PARTY_NUMBER'],
                'Cluster Label': cluster_id,
                'Metric': metric,
                'Metric Description': metric_desc,
                'Value': row[metric],
                'Percentile': perc,
                'Z_score': z,
                'Interpretation': interpretation
            })

df_stat_metrics = pd.DataFrame(final_metrics).round(2)
df_stat_metrics['Is Anomaly'] = df_stat_metrics['PARTY_NUMBER'].map(party_anomaly_dict)
df_stat_metrics = df_stat_metrics.reset_index(drop=True)

In [ ]:
df_stat_metrics[(df_stat_metrics['Is Anomaly'])  & (df_stat_metrics['Percentile'] >= 75)]

In [ ]:
df_stat_metrics[df_stat_metrics['Is Anomaly']  & (df_stat_metrics['Percentile'] >= 75)].iloc[4]['Interpretation']

In [ ]:
#### Peer Density Score

How isolated a party is relative to its nearest peers in behaviour space.

Using unsupervised technique: NearestNeighbors

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

density_results = []
df_peer = df_anom_features.copy()

feature_cols = [c for c in df_peer.columns 
                if c not in ['PARTY_NUMBER', 'Cluster Label', 'Aggregate Anomaly Score', 'Is Anomaly', 
                             'Alert Level', 'cluster_desc', 'mahalanobis',
                             'top_features', 'top_features_contrib_pct', 
                             'mahalanobis_interpretation', 'top_features_percentile']]

for cluster_id, cluster_df in df_peer.groupby('Cluster Label'):
    
    X = cluster_df[feature_cols].fillna(0).values
    
    if len(X) < 5:
        continue

    # scale features to prevent huge distance values
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # use 6 neighbours so we can remove self-distance
    nbrs = NearestNeighbors(n_neighbors=6).fit(X)
    distances, _ = nbrs.kneighbors(X)

    # remove self distance (always 0)
    distances = distances[:, 1:]

    # average distance to local peers
    avg_dist = distances.mean(axis=1)

    # typical peer distance in cluster
    median_dist = np.median(avg_dist)

    for i, row in cluster_df.iterrows():
        
        score = avg_dist[list(cluster_df.index).index(i)] / median_dist if median_dist > 0 else 1
        
        density_results.append({
            'PARTY_NUMBER': row['PARTY_NUMBER'],
            'Cluster Label': cluster_id,
            'Metric': 'Peer Density Score',
            'Value': score,
            'Interpretation': (
                "This party operates in a notably sparse behavioural region compared with nearby peers."
                if score > 2 else
                "This party shows transaction patterns that are materially different from those of nearby peers."
                if score > 1.5 else
                "Party's transaction behaviour patterns are broadly similar to those of most peers."
            )
        })

df_density = pd.DataFrame(density_results)

df_density['Is Anomaly'] = df_density['PARTY_NUMBER'].map(party_anomaly_dict)
df_density = df_density[df_density['Is Anomaly']]

df_density['Value'] = df_density['Value'].round(2)
df_density = df_density.reset_index(drop=True)

In [ ]:
| Peer Density Score | Interpretation |
|--------------------|---------------|
| ~1.0 | Typical peer behaviour (customer sits within a dense group of similar customers) |
| 1.2 – 1.5 | Mild deviation from nearby peers |
| 1.5 – 2.0 | Customer operates in a relatively sparse behavioural region |
| > 2.0 | Strong isolation from peers; behaviour is uncommon within the segment |

In [ ]:
def get_comparison_label(current_month: str, baseline_months: list):
    current_month_fmt = pd.to_datetime(current_month)
    curr_month_name = current_month_fmt.month_name()[:3]
    curr_year = str(current_month_fmt.year)

    baseline_fmt_months = []
    baseline_fmt_years = []

    for month in baseline_months:
        baseline_month_fmt = pd.to_datetime(month)
        baseline_month_name = baseline_month_fmt.month_name()[:3]
        baseline_year = str(baseline_month_fmt.year)
        baseline_fmt_months.append(baseline_month_name)
        baseline_fmt_years.append(str(baseline_year))

    baseline_fmt_years = list(set(baseline_fmt_years))
    total_baseline_months = str(len(baseline_fmt_months))

    return f"{curr_month_name} {curr_year} vs {baseline_fmt_months[0]}-{baseline_fmt_months[1]} {baseline_fmt_years[0]} ({total_baseline_months}MoM)"


In [ ]:
# Initial params
if dataset_type == 'train':
    current_month = '2025-12'
    baseline_months = ['2025-11', '2025-10']
else:
    current_month = '2026-01'
    baseline_months = ['2025-12', '2025-11']
    
comparison_label = get_comparison_label(current_month, baseline_months)

# cnsider only alerted parties
alerted_parties = set(
    df_anom_features.loc[df_anom_features['Is Anomaly'] == True, 'PARTY_NUMBER']
)

# derive year-month
df_tms['TXN_MONTH'] = pd.to_datetime(df_tms['TRANSACTION_DATE']).dt.to_period('M').astype(str)

# Filter current & baseline
df_current = df_tms[df_tms['TXN_MONTH'] == current_month]
df_baseline = df_tms[df_tms['TXN_MONTH'].isin(baseline_months)]

# Aggregate metrics
def aggregate_metrics(df):
    return df.groupby(
        ['PARTY_NUMBER', 'Cluster Label',
         'PRIMARY_MEDIUM_DESC', 'TRANSACTION_CDI_DESC']
    ).agg(
        txn_count=('TXN_REF_NUMBER', 'count'),
        txn_amount=('CURRENCY_AMOUNT', 'sum'),
        avg_amount=('CURRENCY_AMOUNT', 'mean')
    ).reset_index()

curr_agg = aggregate_metrics(df_current)
base_agg = aggregate_metrics(df_baseline)

# Merge current and baseline
merged = curr_agg.merge(
    base_agg,
    on=['PARTY_NUMBER', 'Cluster Label',
        'PRIMARY_MEDIUM_DESC', 'TRANSACTION_CDI_DESC'],
    suffixes=('_curr','_base'),
    how='left'
)

merged[['txn_count_base','txn_amount_base','avg_amount_base']] = \
    merged[['txn_count_base','txn_amount_base','avg_amount_base']].fillna(0)

# Build behavioural features
records = []

for idx, row in merged.iterrows():
    if row['PARTY_NUMBER'] not in alerted_parties:
        continue
    
    features = [
        ('Count', row['txn_count_curr'], row['txn_count_base']),
        ('Amount', row['txn_amount_curr'], row['txn_amount_base']),
        ('Average Amount', row['avg_amount_curr'], row['avg_amount_base'])
    ]
    
    for metric_type, curr_val, base_val in features:
        if base_val == 0 or curr_val <= base_val:
            continue
        
        pct_change = ((curr_val - base_val) / base_val) * 100
        
        feature_name_1 = (
            f"{row['PRIMARY_MEDIUM_DESC']} "
            f"{row['TRANSACTION_CDI_DESC'].title()}"
        )

        feature_name_2 = (
            f"{row['PRIMARY_MEDIUM_DESC']} "
            f"{row['TRANSACTION_CDI_DESC'].lower()}"
        )
        
        records.append({
            'PARTY_NUMBER': row['PARTY_NUMBER'],
            'Cluster Label': row['Cluster Label'],
            'Feature': feature_name_1,
            'Metric Type': metric_type,
            'Actual Value': round(curr_val, 2),
            'Baseline Value': round(base_val, 2),
            'Pct Change': round(pct_change, 1),
            'Comparison Window': comparison_label,
            'Interpretation': (
                f"{feature_name_2} {metric_type.lower()} increased by "
                f"{pct_change:.1f}% compared to the prior two months."
            )
        })

df_behavioural_metrics = pd.DataFrame(records)

In [ ]:
def aggregate_counterparty_metrics(df):
    return df.groupby(['PARTY_NUMBER', 'Cluster Label']).agg(
        unique_counterparties=('BENEFICIARY_PARTY_NO', 'nunique'),
        unique_countries=('COUNTER_PARTY_COUNTRY', 'nunique')
    ).reset_index()

curr_cp = aggregate_counterparty_metrics(df_current)
base_cp = aggregate_counterparty_metrics(df_baseline)

merged_cp = curr_cp.merge(
    base_cp,
    on=['PARTY_NUMBER', 'Cluster Label'],
    suffixes=('_curr', '_base'),
    how='left'
).fillna(0)

records_cp = []

for idx, row in merged_cp.iterrows():
    if row['PARTY_NUMBER'] not in alerted_parties:
        continue

    features = [
        ('Number of counterparties', 
        row['unique_counterparties_curr'], row['unique_counterparties_base']),
        
        ('Geographic spread of counterparties', 
        row['unique_countries_curr'], row['unique_countries_base'])
    ]

    for metric_name, curr_val, base_val in features:
        if base_val == 0 or curr_val <= base_val:
            continue

        pct_change = ((curr_val - base_val) / base_val) * 100

        records_cp.append({
            'PARTY_NUMBER': row['PARTY_NUMBER'],
            'Cluster Label': row['Cluster Label'],
            'Feature': metric_name,
            'Metric Type': 'Count',
            'Actual Value': int(curr_val),
            'Baseline Value': int(base_val),
            'Pct Change': round(pct_change, 1),
            'Comparison Window': comparison_label,
            'Interpretation': (
                f"{metric_name} increased by {pct_change:.1f}% compared to the prior two months."
            )
        })

In [ ]:
df_behavioural_metrics = pd.concat([df_behavioural_metrics, pd.DataFrame(records_cp)], ignore_index=True)
df_behavioural_metrics['Is Anomaly'] = df_behavioural_metrics['PARTY_NUMBER'].map(party_anomaly_dict)
df_behavioural_metrics = df_behavioural_metrics[df_behavioural_metrics['Is Anomaly']]
df_behavioural_metrics = df_behavioural_metrics.reset_index(drop=True)

df_density = df_density[df_density['Is Anomaly']]
df_stat_metrics = df_stat_metrics[df_stat_metrics['Is Anomaly']]

In [ ]:
## Stahel-Donoho Outlyingness (Cluster-wise Scaled Interpretation)

from sklearn.preprocessing import RobustScaler

np.random.seed(42)

outlyingness_results = []
df_clus = df_anom_features.copy()

# Define features to include in SDO calculation
feature_cols = [
    c for c in df_clus.columns
    if c not in [
        'PARTY_NUMBER', 'Cluster Label', 'Aggregate Anomaly Score',
        'Is Anomaly', 'Alert Level', 'cluster_desc', 'mahalanobis',
        'mahalanobis_percentile','top_features','top_features_contrib_pct',
        'mahalanobis_interpretation','top_features_percentile'
    ]
]

n_directions = 500  # Number of random projection directions

for cluster_id, cluster_df in df_clus.groupby('Cluster Label'):
    
    X = cluster_df[feature_cols].fillna(0).values

    # Skip small clusters
    if len(X) < 10:
        continue

    # Robust scaling per cluster to stabilize projections
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    n_samples, n_features = X_scaled.shape
    max_outlyingness = np.zeros(n_samples)

    # Compute Stahel-Donoho Outlyingness via random projections
    for _ in range(n_directions):
        direction = np.random.normal(size=n_features)
        direction = direction / np.linalg.norm(direction)
        proj = X_scaled @ direction
        med = np.median(proj)
        mad = np.median(np.abs(proj - med))
        mad = max(mad, 1e-6)  # Stability floor
        outlyingness = np.abs(proj - med) / mad
        max_outlyingness = np.maximum(max_outlyingness, outlyingness)

    raw_scores = max_outlyingness

    # ------------------------------
    # Cluster-wise min-max scaling
    # ------------------------------
    s_min, s_max = raw_scores.min(), raw_scores.max()
    if s_max - s_min > 0:
        scaled_scores = (raw_scores - s_min) / (s_max - s_min)
    else:
        scaled_scores = np.zeros_like(raw_scores)  # All same values

    # Optional percentile for reference
    #percentiles = np.argsort(np.argsort(raw_scores)) / len(raw_scores) * 100

    # ------------------------------
    # Assign interpretation dynamically
    # ------------------------------
    for idx, row in enumerate(cluster_df.itertuples()):
        raw_score = round(float(raw_scores[idx]), 3)
        scaled_score = round(float(scaled_scores[idx]), 3)
        #percentile_score = round(float(percentiles[idx]), 2)

        # Interpretation based on scaled score
        if scaled_score <= 0.2:
            interp = f"Normalized SDO score {scaled_score:.2f} indicates typical behaviour relative to cluster peers. Customer behaviour aligns closely with peers in this segment."
        elif scaled_score <= 0.4:
            interp = f"Normalized SDO score {scaled_score:.2f} indicates mild deviation from cluster norms. Customer behaviour shows slight differences compared with peer activity."
        elif scaled_score <= 0.6:
            interp = f"Normalized SDO score {scaled_score:.2f} indicates noticeable deviation. Customer behaviour is moderately unusual relative to cluster peers."
        elif scaled_score <= 0.8:
            interp = f"Normalized SDO score {scaled_score:.2f} indicates strong deviation from typical cluster patterns. Customer behaviour is highly unusual compared to peers."
        else:
            interp = f"Normalized SDO score {scaled_score:.2f} indicates extreme deviation from cluster norms. Customer behaviour is extremely anomalous relative to peers and may warrant further review."

        outlyingness_results.append({
            "PARTY_NUMBER": row.PARTY_NUMBER,
            "Cluster Label": cluster_id,
            "Metric": "Stahel-Donoho Outlyingness",
            "Outlyingness": np.round(raw_score, 2),
            "Normalized Outlyingness": np.round(scaled_score, 2),
            #"Percentile": percentile_score,
            "Interpretation": interp
        })

# Create final dataframe
df_stahel = pd.DataFrame(outlyingness_results)

# Attach anomaly flag
df_stahel["Is Anomaly"] = df_stahel["PARTY_NUMBER"].map(party_anomaly_dict)

# Show only anomaly customers
df_stahel = df_stahel[df_stahel["Is Anomaly"] == True].copy()
df_stahel = df_stahel.reset_index(drop=True)

In [ ]:
## Histogram Based Outlier Score (HBOS)

from pyod.models.hbos import HBOS

hbos_results = []

df_hbos = df_anom_features.copy()

feature_cols = [
    c for c in df_hbos.columns
    if c not in [
        'PARTY_NUMBER','Cluster Label','Aggregate Anomaly Score',
        'Is Anomaly','Alert Level','cluster_desc','mahalanobis',
        'mahalanobis_percentile','top_features','top_features_contrib_pct',
        'mahalanobis_interpretation','top_features_percentile'
    ]
]

for cluster_id, cluster_df in df_hbos.groupby('Cluster Label'):

    X = cluster_df[feature_cols].fillna(0).values

    if len(X) < 10:
        continue

    # Fit HBOS
    hbos = HBOS(n_bins=15, contamination=0.02)
    hbos.fit(X)

    scores = hbos.decision_scores_

    # Robust distribution statistics
    med = np.median(scores)

    # median absolute deviation
    mad = np.median(np.abs(scores - med))

    mad = mad if mad > 0 else 1e-6

    for i, row in enumerate(cluster_df.itertuples()):

        score = float(scores[i])

        # deviation from cluster distribution
        deviation = (score - med) / mad

        deviation_abs = abs(deviation)

        # Dynamic interpretation
        if deviation <= 1:
            interp = (
                f"HBOS score {score:.2f} falls within the normal range of behaviour observed "
                f"among peer customers in this cluster, indicating transaction patterns "
                f"consistent with the segment."
            )

        elif deviation <= 1.5:
            interp = (
                f"HBOS score {score:.2f} indicates mild deviation from typical peer behaviour "
                f"in this cluster, suggesting moderately uncommon transaction characteristics."
            )

        elif deviation <= 3:
            interp = (
                f"HBOS score {score:.2f} indicates noticeable deviation from the majority of "
                f"peer customers in this segment, suggesting relatively uncommon transaction patterns."
            )

        elif deviation <= 5:
            interp = (
                f"HBOS score {score:.2f} suggests materially unusual transaction behaviour "
                f"compared with similar customers in this cluster."
            )

        else:
            interp = (
                f"HBOS score {score:.2f} is significantly higher than the typical range "
                f"observed among peer customers in this segment, indicating highly unusual "
                f"transaction characteristics that may warrant further investigation."
            )

        hbos_results.append({
            "PARTY_NUMBER": row.PARTY_NUMBER,
            "Cluster Label": cluster_id,
            "Metric": "Histogram-Based Outlier Score",
            "Value": round(score, 2),
            "Interpretation": interp
        })

df_hbos = pd.DataFrame(hbos_results)

# attach anomaly flag
df_hbos["Is Anomaly"] = df_hbos["PARTY_NUMBER"].map(party_anomaly_dict)

# show only anomaly customers
df_hbos = df_hbos[df_hbos["Is Anomaly"] == True].copy()
df_hbos = df_hbos.reset_index(drop=True)

In [ ]:
wd.save_table(df_db_mahalanobis.round(2), "Mahalanobis Distance - Consumer Banking")
wd.save_table(df_stat_metrics, "Transaction Percentile - Consumer Banking")
wd.save_table(df_density, "Peer Density Score - Consumer Banking")
wd.save_table(df_stahel, "Stahel-Donoho Outlyingness - Consumer Banking")
wd.save_table(df_hbos, "Histogram-Based Outlier Score - Consumer Banking")
wd.save_table(df_behavioural_metrics, "Behavioral Metrics - Consumer Banking")

In [ ]:
# Function to write the data to S3 bucket
def save_df_to_s3(dataset: pd.DataFrame, container_name: str, file_path: str):
    buffer = io.BytesIO()
    dataset.to_parquet(buffer, index=False)
    buffer.seek(0)
    wd.store.put_object(container_name=container_name, key=file_path, data=buffer)
    print(f"DataFrame saved to Nimbus S3 Container: {container_name}/{file_path}")

In [ ]:
save_df_to_s3(df_density, 'solytics-saas-nimbus-uno-uat-s3', f'poc_anomaly_detection/consumer_banking/outputs/density_metrics_{dataset_type}.parquet')
save_df_to_s3(df_behavioural_metrics, 'solytics-saas-nimbus-uno-uat-s3', f'poc_anomaly_detection/consumer_banking/outputs/behavior_metrics_{dataset_type}.parquet')
save_df_to_s3(df_stat_metrics, 'solytics-saas-nimbus-uno-uat-s3', f'poc_anomaly_detection/consumer_banking/outputs/txn_percentile_metrics_{dataset_type}.parquet')
save_df_to_s3(df_db_mahalanobis, 'solytics-saas-nimbus-uno-uat-s3', f'poc_anomaly_detection/consumer_banking/outputs/maha_metrics_{dataset_type}.parquet')